In [ ]:
pip install openai PyPDF2 python-docx google-genai

In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

In [ ]:
import os
import PyPDF2
import docx
from google.colab import userdata
from google import genai

# 1. Initialize Gemini Client
client = genai.Client(api_key=userdata.get("GOOGLE_GENAI_API_KEY"))

# 2. Define File Reader Function
def read_file_content(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == '.pdf':
        text = ""
        with open(file_path, "rb") as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text += page.extract_text() or ""
        return text

    elif ext == '.docx':
        doc = docx.Document(file_path)
        text = []
        for para in doc.paragraphs:
            text.append(para.text)
        return "\n".join(text)

    elif ext in ['.txt', '.md', '.json', '.csv']:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
            return file.read()

    else:
        raise ValueError(f"Unsupported file format: {ext}")

# 3. Define ATS Analysis Function
def analyze_resume(text):
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents = f"""
        You are an ATS resume analyzer.
        Analyze the resume below and return ATS score and feedback.

        Resume:
        {text}

        Resume in JSON format with:
        - ATS Score
        - Summary
        - Strengths
        - Weaknesses
        - Areas of Improvement
        - Recommendations

        Rules:
        - ATS Score should be a number between 0 and 100.
        - Output Only JSON
        - No extra text
        """
    )
    return response.text

In [ ]:
# Execute processing and analysis pipeline
try:
    resume_text = read_file_content(file_name)
    result = analyze_resume(resume_text)
    print(result)
except NameError:
    print("Please run the file upload cell first.")
except Exception as e:
    print(f"An error occurred: {e}")